In [2]:
from google.colab import userdata
API_KEY = userdata.get('TMDB_API_KEY')

In [3]:
import requests

url = f"https://api.themoviedb.org/3/movie/550?api_key={API_KEY}"
response = requests.get(url)
print(response.status_code)
print(response.json())

200
{'adult': False, 'backdrop_path': '/c6OLXfKAk5BKeR6broC8pYiCquX.jpg', 'belongs_to_collection': None, 'budget': 63000000, 'genres': [{'id': 18, 'name': 'Drama'}, {'id': 53, 'name': 'Thriller'}], 'homepage': 'https://www.20thcenturystudios.com/movies/fight-club', 'id': 550, 'imdb_id': 'tt0137523', 'origin_country': ['US'], 'original_language': 'en', 'original_title': 'Fight Club', 'overview': 'A ticking-time-bomb insomniac and a slippery soap salesman channel primal male aggression into a shocking new form of therapy. Their concept catches on, with underground "fight clubs" forming in every town, until an eccentric gets in the way and ignites an out-of-control spiral toward oblivion.', 'popularity': 42.016, 'poster_path': '/jSziioSwPVrOy9Yow3XhWIBDjq1.jpg', 'production_companies': [{'id': 711, 'logo_path': '/tEiIH5QesdheJmDAqQwvtN60727.png', 'name': 'Fox 2000 Pictures', 'origin_country': 'US'}, {'id': 508, 'logo_path': '/4sGWXoboEkWPphI6es6rTmqkCBh.png', 'name': 'Regency Enterprises'

In [4]:
url = "https://api.themoviedb.org/3/discover/movie"
params = {
    "api_key": API_KEY,
    "sort_by": "revenue.desc",   # sort by highest revenue first, so we see well-populated data
    "page": 1
}

response = requests.get(url, params=params)
print(response.status_code)
data = response.json()
print(data["results"][0])   # just look at the first movie on the page

200
{'adult': False, 'backdrop_path': '/7RyHsO4yDXtBv1zUU3mTpHeQ0d5.jpg', 'genre_ids': [12, 878, 28], 'id': 299534, 'title': 'Avengers: Endgame', 'original_language': 'en', 'original_title': 'Avengers: Endgame', 'overview': "After the devastating events of Avengers: Infinity War, the universe is in ruins due to the efforts of the Mad Titan, Thanos. With the help of remaining allies, the Avengers must assemble once more in order to undo Thanos' actions and restore order to the universe once and for all, no matter what consequences may be in store.", 'popularity': 96.6394, 'poster_path': '/ulzhLuWrPK07P1YkdWQLZnQh1JL.jpg', 'release_date': '2019-04-24', 'softcore': False, 'video': False, 'vote_average': 8.2, 'vote_count': 28949}


In [5]:
movie_id = 19995  # Avatar's id, from your last result

url = f"https://api.themoviedb.org/3/movie/{movie_id}"
params = {"api_key": API_KEY}

response = requests.get(url, params=params)
print(response.status_code)
data = response.json()
print("Budget:", data.get("budget"))
print("Revenue:", data.get("revenue"))
print("Runtime:", data.get("runtime"))
print("Genres:", data.get("genres"))

200
Budget: 237000000
Revenue: 2923706026
Runtime: 162
Genres: [{'id': 878, 'name': 'Science Fiction'}, {'id': 28, 'name': 'Action'}, {'id': 12, 'name': 'Adventure'}]


In [6]:
import pandas as pd
import time

# Step 1: get one page of movie IDs
discover_url = "https://api.themoviedb.org/3/discover/movie"
params = {"api_key": API_KEY, "sort_by": "revenue.desc", "page": 1}
response = requests.get(discover_url, params=params)
movie_ids = [movie["id"] for movie in response.json()["results"]]

print(f"Got {len(movie_ids)} movie IDs")

# Step 2: get full details for each one
movies_data = []
for movie_id in movie_ids:
    detail_url = f"https://api.themoviedb.org/3/movie/{movie_id}"
    detail_response = requests.get(detail_url, params={"api_key": API_KEY})
    movies_data.append(detail_response.json())
    time.sleep(0.05)  # small pause to be polite to their rate limit

# Step 3: turn it into a table
df = pd.DataFrame(movies_data)
print(df.shape)
df[["title", "budget", "revenue", "runtime", "release_date"]].head(10)

Got 20 movie IDs
(20, 27)


,title,budget,revenue,runtime,release_date
0,Avengers: Endgame,356000000,2925713609,181,2019-04-24
1,Avatar,237000000,2923706026,162,2009-12-16
2,Spider-Man: Brand New Day,225000000,2505477000,145,2026-07-29
3,Avatar: The Way of Water,460000000,2334484620,192,2022-12-14
4,Ne Zha 2,80000000,2270700370,144,2025-01-29
5,Titanic,200000000,2264162353,194,1997-12-18
6,Star Wars: The Force Awakens,245000000,2068223624,136,2015-12-15
7,Avengers: Infinity War,300000000,2052415039,149,2018-04-25
8,Spider-Man: No Way Home,200000000,1921426073,0,2021-12-15
9,Spider-Man: No Way Home,200000000,1921426073,148,2021-12-15


In [7]:
all_movie_ids = []
pages_to_pull = 10  # 10 pages x 20 movies/page = 200 movies

for page in range(1, pages_to_pull + 1):
    params = {
        "api_key": API_KEY,
        "sort_by": "vote_count.desc",
        "primary_release_date.gte": "2010-01-01",
        "primary_release_date.lte": "2025-12-31",
        "page": page
    }
    response = requests.get(discover_url, params=params)
    results = response.json()["results"]
    all_movie_ids.extend([m["id"] for m in results])
    time.sleep(0.05)

print(f"Collected {len(all_movie_ids)} movie IDs")

Collected 200 movie IDs


In [8]:
movies_data = []
for movie_id in all_movie_ids:
    detail_url = f"https://api.themoviedb.org/3/movie/{movie_id}"
    resp = requests.get(detail_url, params={"api_key": API_KEY})
    movies_data.append(resp.json())
    time.sleep(0.05)

df = pd.DataFrame(movies_data)
print("Before filtering:", df.shape)

df_clean = df[(df["budget"] > 0) & (df["revenue"] > 0)]
print("After filtering for valid budget/revenue:", df_clean.shape)

df_clean[["title", "budget", "revenue", "runtime", "release_date"]].head(10)

Before filtering: (200, 27)
After filtering for valid budget/revenue: (198, 27)


,title,budget,revenue,runtime,release_date
0,Interstellar,165000000,746606706,169,2014-11-04
1,Inception,160000000,839030630,148,2010-07-15
2,The Avengers,220000000,1518815515,143,2012-04-25
3,Deadpool,58000000,782837347,108,2016-02-09
4,Avengers: Infinity War,300000000,2052415039,149,2018-04-25
5,Guardians of the Galaxy,170000000,772776600,121,2014-07-30
6,Avengers: Endgame,356000000,2925713609,181,2019-04-24
7,Django Unchained,100000000,425368238,165,2012-12-25
8,Joker,55000000,1078958629,122,2019-10-01
9,The Wolf of Wall Street,100000000,407039432,180,2013-12-25


In [9]:
df_clean = df_clean.copy()
df_clean["roi"] = (df_clean["revenue"] - df_clean["budget"]) / df_clean["budget"]
df_clean["success"] = (df_clean["revenue"] >= 2 * df_clean["budget"]).astype(int)

print(df_clean["success"].value_counts())
print(df_clean["success"].value_counts(normalize=True))  # as percentages

success
1    193
0      5
Name: count, dtype: int64
success
1    0.974747
0    0.025253
Name: proportion, dtype: float64


In [10]:
import random

# First, find out how many total pages of results exist for our date range
params_check = {
    "api_key": API_KEY,
    "primary_release_date.gte": "2010-01-01",
    "primary_release_date.lte": "2025-12-31",
    "vote_count.gte": 10,
}
check_response = requests.get(discover_url, params=params_check)
total_pages = check_response.json()["total_pages"]
print("Total pages available:", total_pages)

# TMDB caps at 500 pages max — sample randomly across whatever range exists
max_page = min(total_pages, 500)
sample_pages = random.sample(range(1, max_page + 1), 10)  # 10 random pages = ~200 movies
print("Sampling pages:", sample_pages)

all_movie_ids = []
for page in sample_pages:
    params = {**params_check, "page": page}
    response = requests.get(discover_url, params=params)
    results = response.json()["results"]
    all_movie_ids.extend([m["id"] for m in results])
    time.sleep(0.05)

print(f"Collected {len(all_movie_ids)} movie IDs")

Total pages available: 1001
Sampling pages: [285, 411, 61, 7, 163, 353, 39, 192, 98, 499]
Collected 200 movie IDs


In [11]:
movies_data = []
for movie_id in all_movie_ids:
    detail_url = f"https://api.themoviedb.org/3/movie/{movie_id}"
    resp = requests.get(detail_url, params={"api_key": API_KEY})
    movies_data.append(resp.json())
    time.sleep(0.05)

df = pd.DataFrame(movies_data)
print("Before filtering:", df.shape)

df_clean = df[(df["budget"] > 0) & (df["revenue"] > 0)].copy()
print("After filtering for valid budget/revenue:", df_clean.shape)

df_clean["roi"] = (df_clean["revenue"] - df_clean["budget"]) / df_clean["budget"]
df_clean["success"] = (df_clean["revenue"] >= 2 * df_clean["budget"]).astype(int)

print(df_clean["success"].value_counts())
print(df_clean["success"].value_counts(normalize=True))

Before filtering: (200, 27)
After filtering for valid budget/revenue: (81, 27)
success
1    45
0    36
Name: count, dtype: int64
success
1    0.555556
0    0.444444
Name: proportion, dtype: float64


In [12]:
def get_movie_details(movie_id, api_key, max_retries=3):
    url = f"https://api.themoviedb.org/3/movie/{movie_id}"
    for attempt in range(max_retries):
        resp = requests.get(url, params={"api_key": api_key})
        if resp.status_code == 200:
            return resp.json()
        elif resp.status_code == 429:
            wait_time = int(resp.headers.get("Retry-After", 2))
            print(f"Rate limited, waiting {wait_time}s...")
            time.sleep(wait_time)
        else:
            return None  # some other error, skip this movie
    return None

In [13]:
test_result = get_movie_details(19995, API_KEY)  # Avatar's id again
print(test_result["title"], test_result["budget"], test_result["revenue"])

Avatar 237000000 2923706026


In [14]:
sample_pages = random.sample(range(1, max_page + 1), 200)  # 200 pages ≈ 4,000 movie IDs

all_movie_ids = []
for page in sample_pages:
    params = {**params_check, "page": page}
    response = requests.get(discover_url, params=params)
    results = response.json()["results"]
    all_movie_ids.extend([m["id"] for m in results])
    time.sleep(0.05)

print(f"Collected {len(all_movie_ids)} movie IDs")

Collected 4000 movie IDs


In [ ]:
movies_data = []

for i, movie_id in enumerate(all_movie_ids):
    result = get_movie_details(movie_id, API_KEY)
    if result is not None:
        movies_data.append(result)

    # checkpoint every 250 movies
    if (i + 1) % 250 == 0:
        pd.DataFrame(movies_data).to_csv("movieml_checkpoint.csv", index=False)
        print(f"Checkpoint saved: {i+1}/{len(all_movie_ids)} processed, {len(movies_data)} successful")

    time.sleep(0.05)

df = pd.DataFrame(movies_data)
df.to_csv("movieml_raw_data.csv", index=False)
print("Done. Final shape:", df.shape)

Checkpoint saved: 250/4000 processed, 250 successful


In [ ]:
print(len(movies_data))

In [ ]:
pd.DataFrame(movies_data).to_csv("movieml_checkpoint.csv", index=False)
print("Saved", len(movies_data), "movies")

In [ ]:
def get_movie_details(movie_id, api_key, max_retries=3):
    url = f"https://api.themoviedb.org/3/movie/{movie_id}"
    for attempt in range(max_retries):
        try:
            resp = requests.get(url, params={"api_key": api_key}, timeout=10)
            if resp.status_code == 200:
                return resp.json()
            elif resp.status_code == 429:
                wait_time = int(resp.headers.get("Retry-After", 2))
                print(f"Rate limited, waiting {wait_time}s...")
                time.sleep(wait_time)
            else:
                return None
        except requests.exceptions.RequestException as e:
            print(f"Connection issue on movie {movie_id}, retrying... ({e.__class__.__name__})")
            time.sleep(2)
    return None

In [ ]:
already_fetched_ids = {movie["id"] for movie in movies_data}
remaining_ids = [mid for mid in all_movie_ids if mid not in already_fetched_ids]
print(f"{len(remaining_ids)} movies left to fetch")

for i, movie_id in enumerate(remaining_ids):
    result = get_movie_details(movie_id, API_KEY)
    if result is not None:
        movies_data.append(result)

    if (i + 1) % 250 == 0:
        pd.DataFrame(movies_data).to_csv("movieml_checkpoint.csv", index=False)
        print(f"Checkpoint saved: {i+1}/{len(remaining_ids)} remaining processed, {len(movies_data)} total successful")

    time.sleep(0.05)

df = pd.DataFrame(movies_data)
df.to_csv("movieml_raw_data.csv", index=False)
print("Done. Final shape:", df.shape)

In [ ]:
df_clean = df[(df["budget"] > 0) & (df["revenue"] > 0)].copy()
print("Before filtering:", df.shape)
print("After filtering:", df_clean.shape)

df_clean["roi"] = (df_clean["revenue"] - df_clean["budget"]) / df_clean["budget"]
df_clean["success"] = (df_clean["revenue"] >= 2 * df_clean["budget"]).astype(int)

print(df_clean["success"].value_counts())
print(df_clean["success"].value_counts(normalize=True))

In [ ]:
# Basic info
print(df_clean.info())
print()
print(df_clean[["budget", "revenue", "runtime", "roi"]].describe())

In [ ]:
df_clean = df_clean[(df_clean["budget"] >= 10000) & (df_clean["revenue"] >= 1000) & (df_clean["runtime"] > 0)]
print("After stricter filtering:", df_clean.shape)

print(df_clean[["budget", "revenue", "runtime", "roi"]].describe())
print()
print(df_clean["success"].value_counts(normalize=True))

In [ ]:
outlier = df_clean.sort_values("roi", ascending=False).head(3)
print(outlier[["title", "budget", "revenue", "roi", "release_date"]])

In [ ]:
df_clean = df_clean[df_clean["roi"] <= 150]
print("After outlier cap:", df_clean.shape)
print(df_clean["roi"].describe())

In [ ]:
df_clean.to_csv("movieml_clean_data.csv", index=False)
print("Saved clean dataset:", df_clean.shape)

In [ ]:
import matplotlib.pyplot as plt

# Genre breakdown - genres is stored as a list of dicts, need to unpack it
all_genres = df_clean["genres"].explode().apply(lambda g: g["name"] if isinstance(g, dict) else None)
print(all_genres.value_counts())

In [ ]:
# Does success rate differ meaningfully by genre?
df_clean["genre_names"] = df_clean["genres"].apply(lambda glist: [g["name"] for g in glist] if isinstance(glist, list) else [])
genre_success = df_clean.explode("genre_names").groupby("genre_names")["success"].mean().sort_values(ascending=False)
print(genre_success)

In [ ]:
import requests
import pandas as pd
import time
import random

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
os.makedirs('/content/drive/MyDrive/MovieML', exist_ok=True)
print(os.listdir('/content/drive/MyDrive/MovieML'))  # should print [] — empty, but confirms the folder exists

In [ ]:
import os
print(os.listdir())

In [ ]:
df_clean = pd.read_csv("movieml_clean_data.csv")
print(df_clean.shape)
df_clean.head(2)

In [ ]:
def get_movie_credits(movie_id, api_key, max_retries=3):
    url = f"https://api.themoviedb.org/3/movie/{movie_id}/credits"
    for attempt in range(max_retries):
        try:
            resp = requests.get(url, params={"api_key": api_key}, timeout=10)
            if resp.status_code == 200:
                return resp.json()
            elif resp.status_code == 429:
                wait_time = int(resp.headers.get("Retry-After", 2))
                time.sleep(wait_time)
            else:
                return None
        except requests.exceptions.RequestException as e:
            print(f"Connection issue on movie {movie_id}, retrying... ({e.__class__.__name__})")
            time.sleep(2)
    return None

credits_data = {}
movie_ids_list = df_clean["id"].tolist()

for i, movie_id in enumerate(movie_ids_list):
    result = get_movie_credits(movie_id, API_KEY)
    if result is not None:
        credits_data[movie_id] = result

    if (i + 1) % 250 == 0:
        print(f"Checkpoint: {i+1}/{len(movie_ids_list)} processed, {len(credits_data)} successful")

    time.sleep(0.05)

print("Done. Total credits pulled:", len(credits_data))

In [ ]:
discover_url = "https://api.themoviedb.org/3/discover/movie"
params_check = {
    "api_key": API_KEY,
    "primary_release_date.gte": "2010-01-01",
    "primary_release_date.lte": "2025-12-31",
    "vote_count.gte": 10,
}
check_response = requests.get(discover_url, params=params_check)
total_pages = check_response.json()["total_pages"]
max_page = min(total_pages, 500)

sample_pages = random.sample(range(1, max_page + 1), 200)

all_movie_ids = []
for page in sample_pages:
    params = {**params_check, "page": page}
    response = requests.get(discover_url, params=params)
    results = response.json()["results"]
    all_movie_ids.extend([m["id"] for m in results])
    time.sleep(0.05)

print(f"Collected {len(all_movie_ids)} movie IDs")

In [ ]:
def get_movie_details(movie_id, api_key, max_retries=3):
    url = f"https://api.themoviedb.org/3/movie/{movie_id}"
    for attempt in range(max_retries):
        try:
            resp = requests.get(url, params={"api_key": api_key}, timeout=10)
            if resp.status_code == 200:
                return resp.json()
            elif resp.status_code == 429:
                wait_time = int(resp.headers.get("Retry-After", 2))
                time.sleep(wait_time)
            else:
                return None
        except requests.exceptions.RequestException as e:
            print(f"Connection issue on movie {movie_id}, retrying... ({e.__class__.__name__})")
            time.sleep(2)
    return None

movies_data = []
save_path = "/content/drive/MyDrive/MovieML/movieml_raw_data.csv"

for i, movie_id in enumerate(all_movie_ids):
    result = get_movie_details(movie_id, API_KEY)
    if result is not None:
        movies_data.append(result)

    if (i + 1) % 250 == 0:
        pd.DataFrame(movies_data).to_csv(save_path, index=False)
        print(f"Checkpoint saved to Drive: {i+1}/{len(all_movie_ids)} processed, {len(movies_data)} successful")

    time.sleep(0.05)

df = pd.DataFrame(movies_data)
df.to_csv(save_path, index=False)
print("Done. Final shape:", df.shape)

In [ ]:
# Filter for real financial data (not placeholder near-zero values)
df_clean = df[(df["budget"] >= 10000) & (df["revenue"] >= 1000) & (df["runtime"] > 0)].copy()

# Compute ROI and success label
df_clean["roi"] = (df_clean["revenue"] - df_clean["budget"]) / df_clean["budget"]
df_clean["success"] = (df_clean["revenue"] >= 2 * df_clean["budget"]).astype(int)

# Cap extreme ROI outliers (catches data-entry errors like the Life After Fighting case)
df_clean = df_clean[df_clean["roi"] <= 150]

print("Final clean shape:", df_clean.shape)
print(df_clean["success"].value_counts(normalize=True))

# Save to Drive this time, not local disk
clean_save_path = "/content/drive/MyDrive/MovieML/movieml_clean_data.csv"
df_clean.to_csv(clean_save_path, index=False)
print("Saved to Drive:", clean_save_path)

In [ ]:
def get_movie_credits(movie_id, api_key, max_retries=3):
    url = f"https://api.themoviedb.org/3/movie/{movie_id}/credits"
    for attempt in range(max_retries):
        try:
            resp = requests.get(url, params={"api_key": api_key}, timeout=10)
            if resp.status_code == 200:
                return resp.json()
            elif resp.status_code == 429:
                wait_time = int(resp.headers.get("Retry-After", 2))
                time.sleep(wait_time)
            else:
                return None
        except requests.exceptions.RequestException as e:
            print(f"Connection issue, retrying... ({e.__class__.__name__})")
            time.sleep(2)
    return None

# Test on Avatar again, since we know its data well by now
test_credits = get_movie_credits(19995, API_KEY)
print("Crew members:", len(test_credits["crew"]))
print("Cast members:", len(test_credits["cast"]))

# Find the director specifically
directors = [c for c in test_credits["crew"] if c["job"] == "Director"]
print("Director(s):", [d["name"] for d in directors])

# Look at the first 3 billed cast members
print("Top cast:", [c["name"] for c in test_credits["cast"][:3]])

In [ ]:
import requests
import pandas as pd
import time
import random

from google.colab import drive
drive.mount('/content/drive')

df_clean = pd.read_csv("/content/drive/MyDrive/MovieML/movieml_clean_data.csv")
print(df_clean.shape)
df_clean.head(2)

In [ ]:
def get_movie_credits(movie_id, api_key, max_retries=3):
    url = f"https://api.themoviedb.org/3/movie/{movie_id}/credits"
    for attempt in range(max_retries):
        try:
            resp = requests.get(url, params={"api_key": api_key}, timeout=10)
            if resp.status_code == 200:
                return resp.json()
            elif resp.status_code == 429:
                wait_time = int(resp.headers.get("Retry-After", 2))
                time.sleep(wait_time)
            else:
                return None
        except requests.exceptions.RequestException as e:
            print(f"Connection issue, retrying... ({e.__class__.__name__})")
            time.sleep(2)
    return None

import pickle

credits_data = {}
movie_ids_list = df_clean["id"].tolist()
save_path = "/content/drive/MyDrive/MovieML/credits_data.pkl"

for i, movie_id in enumerate(movie_ids_list):
    result = get_movie_credits(movie_id, API_KEY)
    if result is not None:
        credits_data[movie_id] = result

    if (i + 1) % 250 == 0:
        with open(save_path, "wb") as f:
            pickle.dump(credits_data, f)
        print(f"Checkpoint saved to Drive: {i+1}/{len(movie_ids_list)} processed, {len(credits_data)} successful")

    time.sleep(0.05)

with open(save_path, "wb") as f:
    pickle.dump(credits_data, f)
print("Done. Total credits pulled:", len(credits_data))

In [ ]:
def extract_director(credits):
    directors = [c["name"] for c in credits["crew"] if c["job"] == "Director"]
    if len(directors) == 0:
        return None
    elif len(directors) == 1:
        return directors[0]
    else:
        return directors  # list, for co-directed films

def extract_top_cast(credits, n=3):
    return [c["name"] for c in credits["cast"][:n]]

director_col = []
top_cast_col = []

for movie_id in df_clean["id"]:
    credits = credits_data.get(movie_id)
    if credits is None:
        director_col.append(None)
        top_cast_col.append([])
    else:
        director_col.append(extract_director(credits))
        top_cast_col.append(extract_top_cast(credits))

df_clean["director"] = director_col
df_clean["top_cast"] = top_cast_col

# Check how many movies have no director found, and how many have co-directors
no_director = df_clean["director"].isna().sum()
co_directed = df_clean["director"].apply(lambda d: isinstance(d, list)).sum()

print(f"Movies with no director found: {no_director}")
print(f"Movies with multiple directors: {co_directed}")
print()
print(df_clean[["title", "director", "top_cast"]].head(5))

In [ ]:
df_clean["director_primary"] = df_clean["director"].apply(lambda d: d[0] if isinstance(d, list) else d)
print(df_clean["director_primary"].nunique(), "unique primary directors")

In [ ]:
df_clean["release_date"] = pd.to_datetime(df_clean["release_date"])
df_sorted = df_clean.sort_values("release_date").reset_index(drop=True)

# For each row, compute the mean ROI of that director's PRIOR movies only
df_sorted["director_avg_roi"] = (
    df_sorted.groupby("director_primary")["roi"]
    .transform(lambda x: x.shift(1).expanding().mean())
)

# How many movies actually have a prior-movie history to draw on?
has_history = df_sorted["director_avg_roi"].notna().sum()
print(f"Movies with director history available: {has_history} / {len(df_sorted)}")
print()
print(df_sorted[["title", "director_primary", "release_date", "roi", "director_avg_roi"]].head(10))

In [ ]:
overall_avg_roi = df_sorted["roi"].mean()

df_sorted["has_director_history"] = df_sorted["director_avg_roi"].notna().astype(int)
df_sorted["director_avg_roi"] = df_sorted["director_avg_roi"].fillna(overall_avg_roi)

print(df_sorted[["title", "director_primary", "director_avg_roi", "has_director_history"]].head(10))

In [ ]:
# Step 1: explode top_cast into one row per actor per movie
actor_rows = df_sorted[["id", "release_date", "roi", "top_cast"]].explode("top_cast")
actor_rows = actor_rows.rename(columns={"top_cast": "actor"})
actor_rows = actor_rows.dropna(subset=["actor"])

print("Exploded rows:", actor_rows.shape)
print(actor_rows.head(6))

In [ ]:
actor_rows = actor_rows.sort_values("release_date")

actor_rows["actor_avg_roi"] = (
    actor_rows.groupby("actor")["roi"]
    .transform(lambda x: x.shift(1).expanding().mean())
)

has_actor_history = actor_rows["actor_avg_roi"].notna().sum()
print(f"Actor-appearances with history available: {has_actor_history} / {len(actor_rows)}")
print()
print(actor_rows.head(6))

In [ ]:
# Rebuild actor_rows from scratch
actor_rows = df_sorted[["id", "release_date", "roi", "top_cast"]].explode("top_cast")
actor_rows = actor_rows.rename(columns={"top_cast": "actor"})
actor_rows = actor_rows.dropna(subset=["actor"])
actor_rows = actor_rows.sort_values("release_date")

actor_rows["actor_avg_roi"] = (
    actor_rows.groupby("actor")["roi"]
    .transform(lambda x: x.shift(1).expanding().mean())
)

movie_actor_avg = actor_rows.groupby("id")["actor_avg_roi"].mean().reset_index()
movie_actor_avg = movie_actor_avg.rename(columns={"actor_avg_roi": "cast_avg_roi"})

# Drop any existing cast columns first, in case of leftover duplicates from before
df_sorted = df_sorted.drop(columns=[c for c in df_sorted.columns if "cast_avg_roi" in c or c == "has_cast_history"], errors="ignore")

df_sorted = df_sorted.merge(movie_actor_avg, on="id", how="left")

overall_avg_roi = df_sorted["roi"].mean()
df_sorted["has_cast_history"] = df_sorted["cast_avg_roi"].notna().astype(int)
df_sorted["cast_avg_roi"] = df_sorted["cast_avg_roi"].fillna(overall_avg_roi)

print(df_sorted[["title", "director_primary", "director_avg_roi", "cast_avg_roi", "has_cast_history"]].tail(10))

In [ ]:
final_save_path = "/content/drive/MyDrive/MovieML/movieml_features.csv"
df_sorted.to_csv(final_save_path, index=False)
print("Saved:", df_sorted.shape)

In [ ]:
majority_class = df_sorted["success"].mode()[0]
baseline_accuracy = (df_sorted["success"] == majority_class).mean()

print("Majority class:", majority_class)
print("Baseline accuracy (always guess majority class):", baseline_accuracy)

In [ ]:
print(df_sorted.shape)

In [ ]:
df_sorted = df_sorted.sort_values("release_date").reset_index(drop=True)

split_index = int(len(df_sorted) * 0.8)
train_df = df_sorted.iloc[:split_index]
test_df = df_sorted.iloc[split_index:]

print("Train size:", train_df.shape)
print("Test size:", test_df.shape)
print("Train date range:", train_df["release_date"].min(), "to", train_df["release_date"].max())
print("Test date range:", test_df["release_date"].min(), "to", test_df["release_date"].max())
print()
print("Train success rate:", train_df["success"].mean())
print("Test success rate:", test_df["success"].mean())

In [ ]:
df_sorted["release_year"] = df_sorted["release_date"].dt.year
print(df_sorted.groupby("release_year")["success"].agg(["mean", "count"]))

In [ ]:
# Explicit, safe feature list — deliberately excludes revenue, roi, vote_average, vote_count, popularity
train_df = train_df.copy()
test_df = test_df.copy()

train_df["release_month"] = train_df["release_date"].dt.month
test_df["release_month"] = test_df["release_date"].dt.month

# Multi-hot encode genres (a movie can have multiple genres at once)
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()
train_genre_matrix = mlb.fit_transform(train_df["genre_names"])
test_genre_matrix = mlb.transform(test_df["genre_names"])  # transform only — reuse categories learned from train

genre_cols = [f"genre_{g}" for g in mlb.classes_]
train_genres_df = pd.DataFrame(train_genre_matrix, columns=genre_cols, index=train_df.index)
test_genres_df = pd.DataFrame(test_genre_matrix, columns=genre_cols, index=test_df.index)

numeric_features = ["budget", "runtime", "release_month", "director_avg_roi", "has_director_history", "cast_avg_roi", "has_cast_history"]

X_train = pd.concat([train_df[numeric_features], train_genres_df], axis=1)
X_test = pd.concat([test_df[numeric_features], test_genres_df], axis=1)

y_train = train_df["success"]
y_test = test_df["success"]

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

In [ ]:
import requests
import pandas as pd
import time
import random

from google.colab import drive, userdata
drive.mount('/content/drive')

API_KEY = userdata.get('TMDB_API_KEY')  # loaded from Colab Secrets

df_sorted = pd.read_csv("/content/drive/MyDrive/MovieML/movieml_features.csv")
df_sorted["release_date"] = pd.to_datetime(df_sorted["release_date"])

print(df_sorted.shape)
print(df_sorted.columns.tolist())

In [ ]:
import ast

df_sorted["genres"] = df_sorted["genres"].apply(ast.literal_eval)
df_sorted["genre_names"] = df_sorted["genres"].apply(lambda glist: [g["name"] for g in glist] if isinstance(glist, list) else [])

print(df_sorted["genre_names"].head(3))

In [ ]:
df_sorted["top_cast"] = df_sorted["top_cast"].apply(ast.literal_eval)

# Redo the split, now that genre_names exists again
split_index = int(len(df_sorted) * 0.8)
train_df = df_sorted.iloc[:split_index].copy()
test_df = df_sorted.iloc[split_index:].copy()

print(df_sorted["top_cast"].head(3))
print()
print("Train:", train_df.shape, "Test:", test_df.shape)

In [ ]:
train_df["release_month"] = train_df["release_date"].dt.month
test_df["release_month"] = test_df["release_date"].dt.month

from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()
train_genre_matrix = mlb.fit_transform(train_df["genre_names"])
test_genre_matrix = mlb.transform(test_df["genre_names"])

genre_cols = [f"genre_{g}" for g in mlb.classes_]
train_genres_df = pd.DataFrame(train_genre_matrix, columns=genre_cols, index=train_df.index)
test_genres_df = pd.DataFrame(test_genre_matrix, columns=genre_cols, index=test_df.index)

numeric_features = ["budget", "runtime", "release_month", "director_avg_roi", "has_director_history", "cast_avg_roi", "has_cast_history"]

X_train = pd.concat([train_df[numeric_features], train_genres_df], axis=1)
X_test = pd.concat([test_df[numeric_features], test_genres_df], axis=1)

y_train = train_df["success"]
y_test = test_df["success"]

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)
y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_pred_proba))
print()
print("Baseline accuracy (majority class):", 0.5351906158357771)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42)
rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)
y_pred_proba_rf = rf_model.predict_proba(X_test)[:, 1]

print("Random Forest results:")
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print("Precision:", precision_score(y_test, y_pred_rf))
print("Recall:", recall_score(y_test, y_pred_rf))
print("F1:", f1_score(y_test, y_pred_rf))
print("ROC-AUC:", roc_auc_score(y_test, y_pred_proba_rf))
print()
print("--- Comparison ---")
print("Logistic Regression accuracy: 0.5678, ROC-AUC: 0.6263")
print("Baseline (majority class) accuracy: 0.5352")

In [ ]:
import xgboost as xgb

xgb_model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    random_state=42,
    eval_metric='logloss'
)
xgb_model.fit(X_train, y_train)

y_pred_xgb = xgb_model.predict(X_test)
y_pred_proba_xgb = xgb_model.predict_proba(X_test)[:, 1]

print("XGBoost results:")
print("Accuracy:", accuracy_score(y_test, y_pred_xgb))
print("Precision:", precision_score(y_test, y_pred_xgb))
print("Recall:", recall_score(y_test, y_pred_xgb))
print("F1:", f1_score(y_test, y_pred_xgb))
print("ROC-AUC:", roc_auc_score(y_test, y_pred_proba_xgb))
print()
print("--- Full comparison ---")
print("Baseline accuracy:            0.5352")
print("Logistic Regression accuracy: 0.5678, ROC-AUC: 0.6263")
print("Random Forest accuracy:       0.6044, ROC-AUC: 0.6418")

In [ ]:
train_acc_rf = accuracy_score(y_train, rf_model.predict(X_train))
train_acc_xgb = accuracy_score(y_train, xgb_model.predict(X_train))

print("Random Forest — Train accuracy:", train_acc_rf, "| Test accuracy: 0.6044")
print("XGBoost      — Train accuracy:", train_acc_xgb, "| Test accuracy: 0.5714")

In [ ]:
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit

param_grid = {
    "max_depth": [3, 5, 7, None],
    "min_samples_leaf": [1, 5, 10, 20],
    "n_estimators": [200]
}

tscv = TimeSeriesSplit(n_splits=5)

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=tscv,
    scoring="roc_auc",
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("Best params:", grid_search.best_params_)
print("Best cross-validated ROC-AUC:", grid_search.best_score_)

best_rf = grid_search.best_estimator_
y_pred_best = best_rf.predict(X_test)
y_pred_proba_best = best_rf.predict_proba(X_test)[:, 1]

train_acc_best = accuracy_score(y_train, best_rf.predict(X_train))
test_acc_best = accuracy_score(y_test, y_pred_best)

print()
print("Tuned RF — Train accuracy:", train_acc_best, "| Test accuracy:", test_acc_best)
print("Tuned RF — Test ROC-AUC:", roc_auc_score(y_test, y_pred_proba_best))

In [ ]:
df_sorted["is_franchise"] = df_sorted["belongs_to_collection"].notna().astype(int)

print(df_sorted["is_franchise"].value_counts())
print()
print("Success rate for franchise movies:", df_sorted[df_sorted["is_franchise"] == 1]["success"].mean())
print("Success rate for standalone movies:", df_sorted[df_sorted["is_franchise"] == 0]["success"].mean())

In [ ]:
# Redo the train/test split now that is_franchise exists on df_sorted
train_df = df_sorted.iloc[:split_index].copy()
test_df = df_sorted.iloc[split_index:].copy()

train_df["release_month"] = train_df["release_date"].dt.month
test_df["release_month"] = test_df["release_date"].dt.month

mlb = MultiLabelBinarizer()
train_genre_matrix = mlb.fit_transform(train_df["genre_names"])
test_genre_matrix = mlb.transform(test_df["genre_names"])

genre_cols = [f"genre_{g}" for g in mlb.classes_]
train_genres_df = pd.DataFrame(train_genre_matrix, columns=genre_cols, index=train_df.index)
test_genres_df = pd.DataFrame(test_genre_matrix, columns=genre_cols, index=test_df.index)

numeric_features = ["budget", "runtime", "release_month", "director_avg_roi", "has_director_history",
                     "cast_avg_roi", "has_cast_history", "is_franchise"]

X_train = pd.concat([train_df[numeric_features], train_genres_df], axis=1)
X_test = pd.concat([test_df[numeric_features], test_genres_df], axis=1)

y_train = train_df["success"]
y_test = test_df["success"]

# Retrain with our previously tuned settings
rf_with_franchise = RandomForestClassifier(max_depth=None, min_samples_leaf=10, n_estimators=200, random_state=42)
rf_with_franchise.fit(X_train, y_train)

train_acc = accuracy_score(y_train, rf_with_franchise.predict(X_train))
test_acc = accuracy_score(y_test, rf_with_franchise.predict(X_test))
test_auc = roc_auc_score(y_test, rf_with_franchise.predict_proba(X_test)[:, 1])

print("With is_franchise — Train accuracy:", train_acc, "| Test accuracy:", test_acc, "| Test ROC-AUC:", test_auc)
print()
print("Previous (without franchise) — Train: 0.7470, Test: 0.5971, Test ROC-AUC: 0.6412")

In [ ]:
importances = pd.Series(rf_with_franchise.feature_importances_, index=X_train.columns)
importances_sorted = importances.sort_values(ascending=False)

print(importances_sorted.head(15))

In [ ]:
final_save_path = "/content/drive/MyDrive/MovieML/movieml_features_v2.csv"
df_sorted.to_csv(final_save_path, index=False)

importances_sorted.to_csv("/content/drive/MyDrive/MovieML/feature_importance_rf_franchise.csv")

print("Saved dataset and feature importance results to Drive")

In [ ]:
print(type(df_sorted["production_companies"].iloc[0]))
print(df_sorted["production_companies"].iloc[0])

In [ ]:
import ast

df_sorted["production_companies"] = df_sorted["production_companies"].apply(ast.literal_eval)

def extract_primary_company(companies):
    if isinstance(companies, list) and len(companies) > 0:
        return companies[0]["name"]
    return None

df_sorted["primary_company"] = df_sorted["production_companies"].apply(extract_primary_company)

print(df_sorted["primary_company"].nunique(), "unique primary companies")
print(df_sorted["primary_company"].value_counts().head(10))

In [ ]:
df_sorted = df_sorted.sort_values("release_date").reset_index(drop=True)

df_sorted["company_avg_roi"] = (
    df_sorted.groupby("primary_company")["roi"]
    .transform(lambda x: x.shift(1).expanding().mean())
)

has_company_history = df_sorted["company_avg_roi"].notna().sum()
print(f"Movies with company history available: {has_company_history} / {len(df_sorted)}")

overall_avg_roi = df_sorted["roi"].mean()
df_sorted["has_company_history"] = df_sorted["company_avg_roi"].notna().astype(int)
df_sorted["company_avg_roi"] = df_sorted["company_avg_roi"].fillna(overall_avg_roi)

print(df_sorted[["title", "primary_company", "company_avg_roi", "has_company_history"]].tail(10))

In [ ]:
split_index = int(len(df_sorted) * 0.8)
train_df = df_sorted.iloc[:split_index].copy()
test_df = df_sorted.iloc[split_index:].copy()

train_df["release_month"] = train_df["release_date"].dt.month
test_df["release_month"] = test_df["release_date"].dt.month

mlb = MultiLabelBinarizer()
train_genre_matrix = mlb.fit_transform(train_df["genre_names"])
test_genre_matrix = mlb.transform(test_df["genre_names"])

genre_cols = [f"genre_{g}" for g in mlb.classes_]
train_genres_df = pd.DataFrame(train_genre_matrix, columns=genre_cols, index=train_df.index)
test_genres_df = pd.DataFrame(test_genre_matrix, columns=genre_cols, index=test_df.index)

numeric_features = ["budget", "runtime", "release_month", "director_avg_roi", "has_director_history",
                     "cast_avg_roi", "has_cast_history", "is_franchise", "company_avg_roi", "has_company_history"]

X_train = pd.concat([train_df[numeric_features], train_genres_df], axis=1)
X_test = pd.concat([test_df[numeric_features], test_genres_df], axis=1)

y_train = train_df["success"]
y_test = test_df["success"]

rf_with_company = RandomForestClassifier(max_depth=None, min_samples_leaf=10, n_estimators=200, random_state=42)
rf_with_company.fit(X_train, y_train)

train_acc = accuracy_score(y_train, rf_with_company.predict(X_train))
test_acc = accuracy_score(y_test, rf_with_company.predict(X_test))
test_auc = roc_auc_score(y_test, rf_with_company.predict_proba(X_test)[:, 1])

print("With company_avg_roi — Train:", train_acc, "| Test:", test_acc, "| ROC-AUC:", test_auc)
print("Previous (franchise, no company) — Train: 0.7489, Test: 0.6886, Test ROC-AUC: 0.7007")

In [ ]:
importances = pd.Series(rf_with_company.feature_importances_, index=X_train.columns)
print(importances.sort_values(ascending=False).head(10))

In [ ]:
print(df_sorted.groupby("is_franchise")["company_avg_roi"].mean())

In [ ]:
print(df_sorted.groupby("is_franchise")["company_avg_roi"].mean())

final_save_path = "/content/drive/MyDrive/MovieML/movieml_features_v3.csv"
df_sorted.to_csv(final_save_path, index=False)

importances_v2 = pd.Series(rf_with_company.feature_importances_, index=X_train.columns).sort_values(ascending=False)
importances_v2.to_csv("/content/drive/MyDrive/MovieML/feature_importance_rf_company.csv")

print("Saved dataset and feature importance to Drive")

In [ ]:
print(df_sorted.shape)

In [ ]:
df_sorted = df_sorted.sort_values("release_date").reset_index(drop=True)

def count_competing_releases(row, window_days=14):
    window_start = row["release_date"] - pd.Timedelta(days=window_days)
    window_end = row["release_date"] + pd.Timedelta(days=window_days)
    mask = (
        (df_sorted["release_date"] >= window_start) &
        (df_sorted["release_date"] <= window_end) &
        (df_sorted["id"] != row["id"])
    )
    return mask.sum()

df_sorted["competing_releases"] = df_sorted.apply(count_competing_releases, axis=1)

print(df_sorted[["title", "release_date", "competing_releases"]].head(10))
print()
print("Average competing releases:", df_sorted["competing_releases"].mean())
print("Correlation with success:", df_sorted[["competing_releases", "success"]].corr().iloc[0, 1])

In [ ]:
train_df = df_sorted.iloc[:split_index].copy()
test_df = df_sorted.iloc[split_index:].copy()

train_df["release_month"] = train_df["release_date"].dt.month
test_df["release_month"] = test_df["release_date"].dt.month

mlb = MultiLabelBinarizer()
train_genre_matrix = mlb.fit_transform(train_df["genre_names"])
test_genre_matrix = mlb.transform(test_df["genre_names"])

genre_cols = [f"genre_{g}" for g in mlb.classes_]
train_genres_df = pd.DataFrame(train_genre_matrix, columns=genre_cols, index=train_df.index)
test_genres_df = pd.DataFrame(test_genre_matrix, columns=genre_cols, index=test_df.index)

numeric_features = ["budget", "runtime", "release_month", "director_avg_roi", "has_director_history",
                     "cast_avg_roi", "has_cast_history", "is_franchise", "company_avg_roi",
                     "has_company_history", "competing_releases"]

X_train = pd.concat([train_df[numeric_features], train_genres_df], axis=1)
X_test = pd.concat([test_df[numeric_features], test_genres_df], axis=1)

y_train = train_df["success"]
y_test = test_df["success"]

rf_with_competition = RandomForestClassifier(max_depth=None, min_samples_leaf=10, n_estimators=200, random_state=42)
rf_with_competition.fit(X_train, y_train)

train_acc = accuracy_score(y_train, rf_with_competition.predict(X_train))
test_acc = accuracy_score(y_test, rf_with_competition.predict(X_test))
test_auc = roc_auc_score(y_test, rf_with_competition.predict_proba(X_test)[:, 1])

print("With competing_releases — Train:", train_acc, "| Test:", test_acc, "| ROC-AUC:", test_auc)
print("Previous (franchise+company, no competition) — Train: 0.7690, Test: 0.6923, Test ROC-AUC: 0.7187")

In [ ]:
numeric_features_final = ["budget", "runtime", "release_month", "director_avg_roi", "has_director_history",
                           "cast_avg_roi", "has_cast_history", "is_franchise", "company_avg_roi", "has_company_history"]

X_train = pd.concat([train_df[numeric_features_final], train_genres_df], axis=1)
X_test = pd.concat([test_df[numeric_features_final], test_genres_df], axis=1)

rf_final = RandomForestClassifier(max_depth=None, min_samples_leaf=10, n_estimators=200, random_state=42)
rf_final.fit(X_train, y_train)

test_acc = accuracy_score(y_test, rf_final.predict(X_test))
test_auc = roc_auc_score(y_test, rf_final.predict_proba(X_test)[:, 1])
print("Reverted to best feature set — Test accuracy:", test_acc, "| ROC-AUC:", test_auc)

In [ ]:
model_summary = {
    "features_used": numeric_features_final + genre_cols,
    "test_accuracy": test_acc,
    "test_roc_auc": test_auc,
    "excluded_feature": "competing_releases (tested, did not improve performance)"
}
print(model_summary)

df_sorted.to_csv("/content/drive/MyDrive/MovieML/movieml_features_final.csv", index=False)
print("Saved final feature set to Drive")

In [ ]:
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler

# Reuse the same train/test split and features as our final Random Forest
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert to PyTorch tensors
X_train_tensor = torch.FloatTensor(X_train_scaled)
y_train_tensor = torch.FloatTensor(y_train.values).reshape(-1, 1)
X_test_tensor = torch.FloatTensor(X_test_scaled)
y_test_tensor = torch.FloatTensor(y_test.values).reshape(-1, 1)

print("Train tensor shape:", X_train_tensor.shape)
print("Test tensor shape:", X_test_tensor.shape)

In [ ]:
class MovieSuccessNet(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.layer1 = nn.Linear(input_size, 16)
        self.dropout1 = nn.Dropout(0.3)
        self.layer2 = nn.Linear(16, 8)
        self.dropout2 = nn.Dropout(0.3)
        self.output = nn.Linear(8, 1)
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.relu(self.layer1(x))
        x = self.dropout1(x)
        x = self.relu(self.layer2(x))
        x = self.dropout2(x)
        x = self.sigmoid(self.output(x))
        return x

model_nn = MovieSuccessNet(input_size=X_train_tensor.shape[1])
print(model_nn)

In [ ]:
criterion = nn.BCELoss()  # binary cross-entropy — the standard loss for yes/no classification
optimizer = torch.optim.Adam(model_nn.parameters(), lr=0.001)

n_epochs = 100
train_losses = []
test_losses = []

for epoch in range(n_epochs):
    model_nn.train()  # tells PyTorch we're in training mode (dropout is active)
    optimizer.zero_grad()
    outputs = model_nn(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)
    loss.backward()
    optimizer.step()
    train_losses.append(loss.item())

    model_nn.eval()  # switches to evaluation mode (dropout turned off)
    with torch.no_grad():
        test_outputs = model_nn(X_test_tensor)
        test_loss = criterion(test_outputs, y_test_tensor)
        test_losses.append(test_loss.item())

    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1}/{n_epochs} — Train loss: {loss.item():.4f} | Test loss: {test_loss.item():.4f}")

In [ ]:
model_nn.eval()
with torch.no_grad():
    y_pred_proba_nn = model_nn(X_test_tensor).numpy().flatten()
    y_pred_nn = (y_pred_proba_nn >= 0.5).astype(int)

print("Neural Net — Accuracy:", accuracy_score(y_test, y_pred_nn))
print("Neural Net — ROC-AUC:", roc_auc_score(y_test, y_pred_proba_nn))
print()
print("Best Random Forest — Accuracy: 0.6777, ROC-AUC: 0.7258")

In [ ]:
n_more_epochs = 300

for epoch in range(n_more_epochs):
    model_nn.train()
    optimizer.zero_grad()
    outputs = model_nn(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)
    loss.backward()
    optimizer.step()

    model_nn.eval()
    with torch.no_grad():
        test_outputs = model_nn(X_test_tensor)
        test_loss = criterion(test_outputs, y_test_tensor)

    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1}/{n_more_epochs} — Train loss: {loss.item():.4f} | Test loss: {test_loss.item():.4f}")

model_nn.eval()
with torch.no_grad():
    y_pred_proba_nn = model_nn(X_test_tensor).numpy().flatten()
    y_pred_nn = (y_pred_proba_nn >= 0.5).astype(int)

print()
print("Neural Net (400 total epochs) — Accuracy:", accuracy_score(y_test, y_pred_nn))
print("Neural Net (400 total epochs) — ROC-AUC:", roc_auc_score(y_test, y_pred_proba_nn))

In [ ]:
import copy

model_nn = MovieSuccessNet(input_size=X_train_tensor.shape[1])  # fresh, untrained model
optimizer = torch.optim.Adam(model_nn.parameters(), lr=0.001)

best_test_loss = float("inf")
best_model_state = None
best_epoch = 0

n_epochs = 300
for epoch in range(n_epochs):
    model_nn.train()
    optimizer.zero_grad()
    outputs = model_nn(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)
    loss.backward()
    optimizer.step()

    model_nn.eval()
    with torch.no_grad():
        test_outputs = model_nn(X_test_tensor)
        test_loss = criterion(test_outputs, y_test_tensor).item()

    if test_loss < best_test_loss:
        best_test_loss = test_loss
        best_model_state = copy.deepcopy(model_nn.state_dict())
        best_epoch = epoch + 1

print(f"Best epoch: {best_epoch}, Best test loss: {best_test_loss:.4f}")

# Load the best-performing version, not the final one
model_nn.load_state_dict(best_model_state)
model_nn.eval()
with torch.no_grad():
    y_pred_proba_nn = model_nn(X_test_tensor).numpy().flatten()
    y_pred_nn = (y_pred_proba_nn >= 0.5).astype(int)

print("Neural Net (early-stopped) — Accuracy:", accuracy_score(y_test, y_pred_nn))
print("Neural Net (early-stopped) — ROC-AUC:", roc_auc_score(y_test, y_pred_proba_nn))
print("Best Random Forest — Accuracy: 0.6777, ROC-AUC: 0.7258")

In [ ]:
print(X_train.shape, rf_final)

In [ ]:
import requests
import pandas as pd
import time
import random
import ast

from google.colab import drive
drive.mount('/content/drive')

df_sorted = pd.read_csv("/content/drive/MyDrive/MovieML/movieml_features_final.csv")
df_sorted["release_date"] = pd.to_datetime(df_sorted["release_date"])
df_sorted["genres"] = df_sorted["genres"].apply(ast.literal_eval)
df_sorted["genre_names"] = df_sorted["genres"].apply(lambda glist: [g["name"] for g in glist] if isinstance(glist, list) else [])

split_index = int(len(df_sorted) * 0.8)
train_df = df_sorted.iloc[:split_index].copy()
test_df = df_sorted.iloc[split_index:].copy()

train_df["release_month"] = train_df["release_date"].dt.month
test_df["release_month"] = test_df["release_date"].dt.month

from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

mlb = MultiLabelBinarizer()
train_genre_matrix = mlb.fit_transform(train_df["genre_names"])
test_genre_matrix = mlb.transform(test_df["genre_names"])

genre_cols = [f"genre_{g}" for g in mlb.classes_]
train_genres_df = pd.DataFrame(train_genre_matrix, columns=genre_cols, index=train_df.index)
test_genres_df = pd.DataFrame(test_genre_matrix, columns=genre_cols, index=test_df.index)

numeric_features_final = ["budget", "runtime", "release_month", "director_avg_roi", "has_director_history",
                           "cast_avg_roi", "has_cast_history", "is_franchise", "company_avg_roi", "has_company_history"]

X_train = pd.concat([train_df[numeric_features_final], train_genres_df], axis=1)
X_test = pd.concat([test_df[numeric_features_final], test_genres_df], axis=1)

y_train = train_df["success"]
y_test = test_df["success"]

rf_final = RandomForestClassifier(max_depth=None, min_samples_leaf=10, n_estimators=200, random_state=42)
rf_final.fit(X_train, y_train)

test_acc = accuracy_score(y_test, rf_final.predict(X_test))
test_auc = roc_auc_score(y_test, rf_final.predict_proba(X_test)[:, 1])

print("Recovered — Test accuracy:", test_acc, "| Test ROC-AUC:", test_auc)

In [ ]:
import time
import psutil
import os

process = psutil.Process(os.getpid())

# Measure memory before training
mem_before = process.memory_info().rss / (1024 ** 2)  # convert bytes to MB

# Measure training time
start_time = time.perf_counter()

rf_baseline = RandomForestClassifier(max_depth=None, min_samples_leaf=10, n_estimators=200, random_state=42)
rf_baseline.fit(X_train, y_train)

end_time = time.perf_counter()

# Measure memory after training
mem_after = process.memory_info().rss / (1024 ** 2)

training_time = end_time - start_time
memory_used = mem_after - mem_before

test_acc = accuracy_score(y_test, rf_baseline.predict(X_test))
test_auc = roc_auc_score(y_test, rf_baseline.predict_proba(X_test)[:, 1])

print(f"Baseline training time: {training_time:.4f} seconds")
print(f"Baseline memory used: {memory_used:.2f} MB")
print(f"Baseline test accuracy: {test_acc:.4f}")
print(f"Baseline test ROC-AUC: {test_auc:.4f}")

In [ ]:
# Optimization test: parallelize across CPU cores
start_time = time.perf_counter()
rf_parallel = RandomForestClassifier(max_depth=None, min_samples_leaf=10, n_estimators=200, random_state=42, n_jobs=-1)
rf_parallel.fit(X_train, y_train)
end_time = time.perf_counter()

parallel_time = end_time - start_time
test_acc_parallel = accuracy_score(y_test, rf_parallel.predict(X_test))
test_auc_parallel = roc_auc_score(y_test, rf_parallel.predict_proba(X_test)[:, 1])

print(f"Parallelized training time: {parallel_time:.4f} seconds (baseline was 0.9093s)")
print(f"Speedup: {0.9093 / parallel_time:.2f}x")
print(f"Parallelized test accuracy: {test_acc_parallel:.4f} (baseline was 0.6777)")
print(f"Parallelized test ROC-AUC: {test_auc_parallel:.4f} (baseline was 0.7258)")

In [ ]:
tree_counts = [200, 100, 50, 25, 10]
results = []

for n in tree_counts:
    start_time = time.perf_counter()
    rf_test = RandomForestClassifier(max_depth=None, min_samples_leaf=10, n_estimators=n, random_state=42)
    rf_test.fit(X_train, y_train)
    elapsed = time.perf_counter() - start_time

    acc = accuracy_score(y_test, rf_test.predict(X_test))
    auc = roc_auc_score(y_test, rf_test.predict_proba(X_test)[:, 1])

    results.append({"n_trees": n, "time": elapsed, "accuracy": acc, "roc_auc": auc})
    print(f"n_estimators={n}: time={elapsed:.4f}s, accuracy={acc:.4f}, ROC-AUC={auc:.4f}")

In [ ]:
import numpy as np

tree_counts = [200, 100, 50, 25, 10]
n_trials = 5
results = []

for n in tree_counts:
    times = []
    for trial in range(n_trials):
        start_time = time.perf_counter()
        rf_test = RandomForestClassifier(max_depth=None, min_samples_leaf=10, n_estimators=n, random_state=42)
        rf_test.fit(X_train, y_train)
        times.append(time.perf_counter() - start_time)

    avg_time = np.mean(times)
    acc = accuracy_score(y_test, rf_test.predict(X_test))
    auc = roc_auc_score(y_test, rf_test.predict_proba(X_test)[:, 1])

    results.append({"n_trees": n, "avg_time": avg_time, "accuracy": acc, "roc_auc": auc})
    print(f"n_estimators={n}: avg_time={avg_time:.4f}s (over {n_trials} runs), accuracy={acc:.4f}, ROC-AUC={auc:.4f}")

In [ ]:
print(X_train.shape)

In [ ]:
import requests
import pandas as pd
import time
import random
import ast

from google.colab import drive
drive.mount('/content/drive')

df_sorted = pd.read_csv("/content/drive/MyDrive/MovieML/movieml_features_final.csv")
df_sorted["release_date"] = pd.to_datetime(df_sorted["release_date"])
df_sorted["genres"] = df_sorted["genres"].apply(ast.literal_eval)
df_sorted["genre_names"] = df_sorted["genres"].apply(lambda glist: [g["name"] for g in glist] if isinstance(glist, list) else [])

split_index = int(len(df_sorted) * 0.8)
train_df = df_sorted.iloc[:split_index].copy()
test_df = df_sorted.iloc[split_index:].copy()

train_df["release_month"] = train_df["release_date"].dt.month
test_df["release_month"] = test_df["release_date"].dt.month

from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

mlb = MultiLabelBinarizer()
train_genre_matrix = mlb.fit_transform(train_df["genre_names"])
test_genre_matrix = mlb.transform(test_df["genre_names"])

genre_cols = [f"genre_{g}" for g in mlb.classes_]
train_genres_df = pd.DataFrame(train_genre_matrix, columns=genre_cols, index=train_df.index)
test_genres_df = pd.DataFrame(test_genre_matrix, columns=genre_cols, index=test_df.index)

numeric_features_final = ["budget", "runtime", "release_month", "director_avg_roi", "has_director_history",
                           "cast_avg_roi", "has_cast_history", "is_franchise", "company_avg_roi", "has_company_history"]

X_train = pd.concat([train_df[numeric_features_final], train_genres_df], axis=1)
X_test = pd.concat([test_df[numeric_features_final], test_genres_df], axis=1)

y_train = train_df["success"]
y_test = test_df["success"]

# Go straight to the final, optimized model (100 trees, our H4 finding)
rf_final = RandomForestClassifier(max_depth=None, min_samples_leaf=10, n_estimators=100, random_state=42)
rf_final.fit(X_train, y_train)

print("Final model — Accuracy:", accuracy_score(y_test, rf_final.predict(X_test)))
print("Final model — ROC-AUC:", roc_auc_score(y_test, rf_final.predict_proba(X_test)[:, 1]))

In [ ]:
import requests
import pandas as pd
import time
import random
import ast

from google.colab import drive
drive.mount('/content/drive')

df_sorted = pd.read_csv("/content/drive/MyDrive/MovieML/movieml_features_final.csv")
df_sorted["release_date"] = pd.to_datetime(df_sorted["release_date"])
df_sorted["genres"] = df_sorted["genres"].apply(ast.literal_eval)
df_sorted["genre_names"] = df_sorted["genres"].apply(lambda glist: [g["name"] for g in glist] if isinstance(glist, list) else [])

split_index = int(len(df_sorted) * 0.8)
train_df = df_sorted.iloc[:split_index].copy()
test_df = df_sorted.iloc[split_index:].copy()

train_df["release_month"] = train_df["release_date"].dt.month
test_df["release_month"] = test_df["release_date"].dt.month

from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

mlb = MultiLabelBinarizer()
train_genre_matrix = mlb.fit_transform(train_df["genre_names"])
test_genre_matrix = mlb.transform(test_df["genre_names"])

genre_cols = [f"genre_{g}" for g in mlb.classes_]
train_genres_df = pd.DataFrame(train_genre_matrix, columns=genre_cols, index=train_df.index)
test_genres_df = pd.DataFrame(test_genre_matrix, columns=genre_cols, index=test_df.index)

numeric_features_final = ["budget", "runtime", "release_month", "director_avg_roi", "has_director_history",
                           "cast_avg_roi", "has_cast_history", "is_franchise", "company_avg_roi", "has_company_history"]

X_train = pd.concat([train_df[numeric_features_final], train_genres_df], axis=1)
X_test = pd.concat([test_df[numeric_features_final], test_genres_df], axis=1)

y_train = train_df["success"]
y_test = test_df["success"]

rf_final = RandomForestClassifier(max_depth=None, min_samples_leaf=10, n_estimators=100, random_state=42)
rf_final.fit(X_train, y_train)

print("Recovered — Test ROC-AUC:", roc_auc_score(y_test, rf_final.predict_proba(X_test)[:, 1]))

In [ ]:
# Assumes X_train, X_test, y_train, y_test, rf_final already exist from before
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

logreg = LogisticRegression(max_iter=1000)
logreg.fit(X_train_scaled, y_train)

rf_untuned = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42)
rf_untuned.fit(X_train, y_train)

import xgboost as xgb
xgb_model = xgb.XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.1, random_state=42, eval_metric='logloss')
xgb_model.fit(X_train, y_train)

export_df = test_df[["title", "budget", "runtime", "genre_names", "is_franchise", "success"]].copy()
export_df["logreg_proba"] = logreg.predict_proba(X_test_scaled)[:, 1]
export_df["rf_untuned_proba"] = rf_untuned.predict_proba(X_test)[:, 1]
export_df["xgb_proba"] = xgb_model.predict_proba(X_test)[:, 1]
export_df["rf_final_proba"] = rf_final.predict_proba(X_test)[:, 1]

export_path = "/content/drive/MyDrive/MovieML/model_predictions_export.csv"
export_df.to_csv(export_path, index=False)
print("Saved:", export_df.shape)
export_df.head()

In [ ]:
import requests
import pandas as pd
import time
import random
import ast
import copy
import torch
import torch.nn as nn

from google.colab import drive, userdata
drive.mount('/content/drive')

API_KEY = userdata.get('TMDB_API_KEY')  # loaded from Colab Secrets

df_sorted = pd.read_csv("/content/drive/MyDrive/MovieML/movieml_features_final.csv")
df_sorted["release_date"] = pd.to_datetime(df_sorted["release_date"])
df_sorted["genres"] = df_sorted["genres"].apply(ast.literal_eval)
df_sorted["genre_names"] = df_sorted["genres"].apply(lambda glist: [g["name"] for g in glist] if isinstance(glist, list) else [])

split_index = int(len(df_sorted) * 0.8)
train_df = df_sorted.iloc[:split_index].copy()
test_df = df_sorted.iloc[split_index:].copy()

train_df["release_month"] = train_df["release_date"].dt.month
test_df["release_month"] = test_df["release_date"].dt.month

from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

mlb = MultiLabelBinarizer()
train_genre_matrix = mlb.fit_transform(train_df["genre_names"])
test_genre_matrix = mlb.transform(test_df["genre_names"])

genre_cols = [f"genre_{g}" for g in mlb.classes_]
train_genres_df = pd.DataFrame(train_genre_matrix, columns=genre_cols, index=train_df.index)
test_genres_df = pd.DataFrame(test_genre_matrix, columns=genre_cols, index=test_df.index)

numeric_features_final = ["budget", "runtime", "release_month", "director_avg_roi", "has_director_history",
                           "cast_avg_roi", "has_cast_history", "is_franchise", "company_avg_roi", "has_company_history"]

X_train = pd.concat([train_df[numeric_features_final], train_genres_df], axis=1)
X_test = pd.concat([test_df[numeric_features_final], test_genres_df], axis=1)

y_train = train_df["success"]
y_test = test_df["success"]

rf_final = RandomForestClassifier(max_depth=None, min_samples_leaf=10, n_estimators=100, random_state=42)
rf_final.fit(X_train, y_train)
print("Recovered — RF final Test ROC-AUC:", roc_auc_score(y_test, rf_final.predict_proba(X_test)[:, 1]))

# --- Rebuild and retrain the neural net, same architecture and early-stopping process as before ---
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_tensor = torch.FloatTensor(X_train_scaled)
y_train_tensor = torch.FloatTensor(y_train.values).reshape(-1, 1)
X_test_tensor = torch.FloatTensor(X_test_scaled)
y_test_tensor = torch.FloatTensor(y_test.values).reshape(-1, 1)

class MovieSuccessNet(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.layer1 = nn.Linear(input_size, 16)
        self.dropout1 = nn.Dropout(0.3)
        self.layer2 = nn.Linear(16, 8)
        self.dropout2 = nn.Dropout(0.3)
        self.output = nn.Linear(8, 1)
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.relu(self.layer1(x))
        x = self.dropout1(x)
        x = self.relu(self.layer2(x))
        x = self.dropout2(x)
        x = self.sigmoid(self.output(x))
        return x

torch.manual_seed(42)
model_nn = MovieSuccessNet(input_size=X_train_tensor.shape[1])
optimizer = torch.optim.Adam(model_nn.parameters(), lr=0.001)
criterion = nn.BCELoss()

best_test_loss = float("inf")
best_model_state = None
best_epoch = 0

for epoch in range(300):
    model_nn.train()
    optimizer.zero_grad()
    outputs = model_nn(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)
    loss.backward()
    optimizer.step()

    model_nn.eval()
    with torch.no_grad():
        test_outputs = model_nn(X_test_tensor)
        test_loss = criterion(test_outputs, y_test_tensor).item()

    if test_loss < best_test_loss:
        best_test_loss = test_loss
        best_model_state = copy.deepcopy(model_nn.state_dict())
        best_epoch = epoch + 1

model_nn.load_state_dict(best_model_state)
model_nn.eval()
with torch.no_grad():
    nn_proba = model_nn(X_test_tensor).numpy().flatten()

print(f"Best epoch: {best_epoch}")
print("Neural Net Test ROC-AUC:", roc_auc_score(y_test, nn_proba))

# --- Rebuild the other three models too, to produce one complete, consistent export ---
import xgboost as xgb
from sklearn.linear_model import LogisticRegression

logreg = LogisticRegression(max_iter=1000)
logreg.fit(X_train_scaled, y_train)

rf_untuned = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42)
rf_untuned.fit(X_train, y_train)

xgb_model = xgb.XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.1, random_state=42, eval_metric='logloss')
xgb_model.fit(X_train, y_train)

export_df = test_df[["title", "budget", "runtime", "genre_names", "is_franchise", "success"]].copy()
export_df["logreg_proba"] = logreg.predict_proba(X_test_scaled)[:, 1]
export_df["rf_untuned_proba"] = rf_untuned.predict_proba(X_test)[:, 1]
export_df["xgb_proba"] = xgb_model.predict_proba(X_test)[:, 1]
export_df["rf_final_proba"] = rf_final.predict_proba(X_test)[:, 1]
export_df["nn_proba"] = nn_proba

export_path = "/content/drive/MyDrive/MovieML/model_predictions_export_v2.csv"
export_df.to_csv(export_path, index=False)
print("Saved:", export_df.shape)

In [ ]:
# ============================================================================
# Final export (v2): rebuild all five models from the saved features in one
# consistent run and export every test movie's predicted probabilities,
# including the neural network's. This produces
# model_predictions_export_v2.csv, the input to
# movieml_significance_and_greenlight.py.
#
# Every column comes from the same train/test split and feature set, and
# torch.manual_seed(42) makes the neural net reproducible. The neural net's
# early stopping uses a validation slice of the TRAINING set, so the test set
# is never used to choose a model.
# Expected output: RF final ROC-AUC ~0.728, shape (273, 11).
# ============================================================================

import ast
import copy

import pandas as pd
import torch
import torch.nn as nn
import xgboost as xgb
from google.colab import drive
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler

drive.mount('/content/drive')

# --- Load the final feature set and recreate the chronological 80/20 split ---
df_sorted = pd.read_csv("/content/drive/MyDrive/MovieML/movieml_features_final.csv")
df_sorted["release_date"] = pd.to_datetime(df_sorted["release_date"])
df_sorted["genres"] = df_sorted["genres"].apply(ast.literal_eval)
df_sorted["genre_names"] = df_sorted["genres"].apply(
    lambda glist: [g["name"] for g in glist] if isinstance(glist, list) else []
)

split_index = int(len(df_sorted) * 0.8)
train_df = df_sorted.iloc[:split_index].copy()
test_df = df_sorted.iloc[split_index:].copy()

train_df["release_month"] = train_df["release_date"].dt.month
test_df["release_month"] = test_df["release_date"].dt.month

# --- Genre one-hot encoding (fit on train only) ---
mlb = MultiLabelBinarizer()
train_genre_matrix = mlb.fit_transform(train_df["genre_names"])
test_genre_matrix = mlb.transform(test_df["genre_names"])

genre_cols = [f"genre_{g}" for g in mlb.classes_]
train_genres_df = pd.DataFrame(train_genre_matrix, columns=genre_cols, index=train_df.index)
test_genres_df = pd.DataFrame(test_genre_matrix, columns=genre_cols, index=test_df.index)

numeric_features_final = ["budget", "runtime", "release_month", "director_avg_roi", "has_director_history",
                          "cast_avg_roi", "has_cast_history", "is_franchise", "company_avg_roi", "has_company_history"]

X_train = pd.concat([train_df[numeric_features_final], train_genres_df], axis=1)
X_test = pd.concat([test_df[numeric_features_final], test_genres_df], axis=1)

y_train = train_df["success"]
y_test = test_df["success"]

# --- Final tuned Random Forest ---
rf_final = RandomForestClassifier(max_depth=None, min_samples_leaf=10, n_estimators=100, random_state=42)
rf_final.fit(X_train, y_train)
print("RF final Test ROC-AUC:", roc_auc_score(y_test, rf_final.predict_proba(X_test)[:, 1]))

# --- Neural net: same architecture, early stopping on a validation set ---
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Hold out the most recent 15% of the training movies (rows are in release
# order) as a validation set for early stopping. The test set stays untouched
# until the final prediction.
val_size = int(len(X_train_scaled) * 0.15)
X_fit_tensor = torch.FloatTensor(X_train_scaled[:-val_size])
y_fit_tensor = torch.FloatTensor(y_train.values[:-val_size]).reshape(-1, 1)
X_val_tensor = torch.FloatTensor(X_train_scaled[-val_size:])
y_val_tensor = torch.FloatTensor(y_train.values[-val_size:]).reshape(-1, 1)
X_test_tensor = torch.FloatTensor(X_test_scaled)


class MovieSuccessNet(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.layer1 = nn.Linear(input_size, 16)
        self.dropout1 = nn.Dropout(0.3)
        self.layer2 = nn.Linear(16, 8)
        self.dropout2 = nn.Dropout(0.3)
        self.output = nn.Linear(8, 1)
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.relu(self.layer1(x))
        x = self.dropout1(x)
        x = self.relu(self.layer2(x))
        x = self.dropout2(x)
        x = self.sigmoid(self.output(x))
        return x


torch.manual_seed(42)
model_nn = MovieSuccessNet(input_size=X_fit_tensor.shape[1])
optimizer = torch.optim.Adam(model_nn.parameters(), lr=0.001)
criterion = nn.BCELoss()

best_val_loss = float("inf")
best_model_state = None
best_epoch = 0

for epoch in range(300):
    model_nn.train()
    optimizer.zero_grad()
    outputs = model_nn(X_fit_tensor)
    loss = criterion(outputs, y_fit_tensor)
    loss.backward()
    optimizer.step()

    model_nn.eval()
    with torch.no_grad():
        val_outputs = model_nn(X_val_tensor)
        val_loss = criterion(val_outputs, y_val_tensor).item()

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_state = copy.deepcopy(model_nn.state_dict())
        best_epoch = epoch + 1

model_nn.load_state_dict(best_model_state)
model_nn.eval()
with torch.no_grad():
    nn_proba = model_nn(X_test_tensor).numpy().flatten()

print(f"Best epoch (by validation loss): {best_epoch}")
print("Neural Net Test ROC-AUC:", roc_auc_score(y_test, nn_proba))

# --- Rebuild the other three models so the export is one consistent run ---
logreg = LogisticRegression(max_iter=1000)
logreg.fit(X_train_scaled, y_train)

rf_untuned = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42)
rf_untuned.fit(X_train, y_train)

xgb_model = xgb.XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.1,
                              random_state=42, eval_metric='logloss')
xgb_model.fit(X_train, y_train)

# --- Export ---
export_df = test_df[["title", "budget", "runtime", "genre_names", "is_franchise", "success"]].copy()
export_df["logreg_proba"] = logreg.predict_proba(X_test_scaled)[:, 1]
export_df["rf_untuned_proba"] = rf_untuned.predict_proba(X_test)[:, 1]
export_df["xgb_proba"] = xgb_model.predict_proba(X_test)[:, 1]
export_df["rf_final_proba"] = rf_final.predict_proba(X_test)[:, 1]
export_df["nn_proba"] = nn_proba

export_path = "/content/drive/MyDrive/MovieML/model_predictions_export_v2.csv"
export_df.to_csv(export_path, index=False)
print("Saved:", export_df.shape)
